# Audio alignment with SGPA

**Goal:** split a recorded sentence into one audio segment per word, with cuts at quiet points, so each word can later be masked as a player.

**Why it matters:** an audio explanation is only auditable when every player is a whole word; a cut inside a sound changes what the model hears and makes attributions irreproducible.

**Prerequisites:** `uv sync --extra audio --extra transformers`. The first run downloads `facebook/wav2vec2-base-960h` (about 360 MB); its load report lists `masked_spec_embed` as missing, a training-only parameter. The clip `data/voices_sp0307.wav` is from the VOiCES corpus (CC BY 4.0).

**Output:** a table of word segments with times, confidence and whether both edges moved to a quiet point.

In [1]:
from pathlib import Path

import numpy as np

from nlp_shap import AudioPart
from nlp_shap.alignment import (
    AudioSegment,
    SpectrogramGuidedAligner,
    Wav2Vec2Ctc,
    decode_audio,
)

## Align a sentence

`Wav2Vec2Ctc` pins the Hub revision so the segments are reproducible. `align` takes the transcript as a string or a token list and the audio as an `AudioPart`.

In [2]:
MODEL_ID = "facebook/wav2vec2-base-960h"
REVISION = "22aad52d435eb6dbaf354bdad9b0da84ce7d6156"
TRANSCRIPT = "I had that curiosity beside me at this moment"

audio = AudioPart(Path("data/voices_sp0307.wav"), sample_rate=16000)
aligner = SpectrogramGuidedAligner(Wav2Vec2Ctc(MODEL_ID, revision=REVISION))
segments = aligner.align(TRANSCRIPT, audio)


def show(rows: list[AudioSegment]) -> None:
    for s in rows:
        print(f"{s.token:10} {s.start_time:6.3f} {s.end_time:6.3f} "
              f"conf={s.confidence:.2f} refined={s.boundary_refined}")


show(segments)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


I           0.523  0.624 conf=1.00 refined=True
had         0.624  0.885 conf=0.97 refined=True
that        0.885  1.078 conf=0.97 refined=True
curiosity   1.078  1.851 conf=0.99 refined=False
beside      1.851  2.273 conf=1.00 refined=False
me          2.273  2.515 conf=1.00 refined=True
at          2.515  2.575 conf=0.99 refined=True
this        2.575  2.780 conf=0.94 refined=True
moment      2.780  3.182 conf=0.98 refined=True


## Boundary refinement

Each boundary moves, inside the gap between two words, to the minimum of `energy_weight * RMS + flux_weight * spectral flux`, and only if that point is quiet. Neighbouring segments share their boundary, so no audio is lost or counted twice. Weighting spectral flux only moves some cuts, but far fewer land on quiet points than with the default weights:

In [3]:
flux_only = SpectrogramGuidedAligner(aligner.model, energy_weight=0.0, flux_weight=1.0)
other = flux_only.align(TRANSCRIPT, audio)
print("shared boundaries:", all(l.end_time == r.start_time for l, r in zip(segments, segments[1:])))
print("start shift (s):", [(a.token, round(b.start_time - a.start_time, 3)) for a, b in zip(segments, other)])
print("refined: default", sum(s.boundary_refined for s in segments), "flux only", sum(s.boundary_refined for s in other))

shared boundaries: True
start shift (s): [('I', 0.0), ('had', 0.0), ('that', -0.03), ('curiosity', -0.012), ('beside', 0.0), ('me', 0.0), ('at', -0.04), ('this', 0.0), ('moment', 0.016)]
refined: default 7 flux only 1


## Segment audio

`attach_audio=True` stores each segment as WAV bytes; this is the unit a masking policy removes or keeps.

In [4]:
with_audio = aligner.align(TRANSCRIPT, audio, attach_audio=True)
word = max(with_audio, key=lambda s: s.duration)
waveform, rate = decode_audio(word.audio)
print(word.token, f"{len(waveform) / rate:.3f} s", len(waveform) == word.end_sample - word.start_sample)

curiosity 0.773 s True


## Guards and custom models

Tokens the model cannot spell get no segment; a transcript with none, or unusable weights, raise `ValueError`. Any object with `sample_rate`, `blank_id`, `vocab` and `log_probs` is a `CtcModel`, for example a fixed matrix in a test.

In [5]:
print([s.token for s in aligner.align("I had — that", audio)][:3])
for call in (lambda: aligner.align("— …", audio),
             lambda: SpectrogramGuidedAligner(aligner.model, energy_weight=0, flux_weight=0)):
    try:
        call()
    except ValueError as error:
        print("ValueError:", error)


class FixedCtc:
    sample_rate, blank_id, vocab = 16000, 0, {"<pad>": 0, "A": 1}

    def log_probs(self, waveform: np.ndarray) -> np.ndarray:
        frames = len(waveform) // 320
        lp = np.full((frames, 2), -20.0)
        lp[:, 0] = 0.0
        lp[frames // 3 : 2 * frames // 3, :] = [-20.0, 0.0]
        return lp


show(SpectrogramGuidedAligner(FixedCtc()).align("a", audio))

['I', 'had', 'that']
ValueError: the transcript has no characters the CTC model knows
ValueError: boundary weights must be non-negative with a positive sum
a           1.080  2.260 conf=1.00 refined=True
